# Historical Simulation: Value at Risk & Expected Shortfall
**Author:** Dev  
**Branch:** `feature/risk-metrics`  
**Project:** Empirical Portfolio Risk Modeling via Monte Carlo Simulation  

---

## Purpose

This notebook implements **historical simulation** to compute two core tail-risk metrics:

- **Value at Risk (VaR)** at the 95% and 99% confidence levels
- **Expected Shortfall (ES)** at the 95% and 99% confidence levels

It also visualises the empirical log-return distribution with VaR threshold lines and saves the figure to `/results/`.  
This notebook is **self-contained**: it simulates one equity's price series if no CSV from Nihan's data pipeline is present, so it can be run from a fresh kernel at any time.

---

## Methodology Overview

Historical simulation is a non-parametric approach to risk estimation. Rather than assuming returns follow a specific distribution (e.g. Normal), we use the **empirical distribution of observed returns** directly. This is particularly valuable in portfolio risk modelling because real equity returns exhibit fat tails and skewness that parametric models underestimate.

The steps are:
1. Compute daily log-returns from closing prices.
2. Sort the empirical return distribution.
3. Read VaR as the relevant quantile of that sorted distribution.
4. Compute ES as the mean of all returns below the VaR threshold.
5. Visualise the distribution with VaR threshold lines overlaid.

## 0. Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ────────────────────────────────────────────────────────────
np.random.seed(42)

# ── Paths (relative to notebook location) ─────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
REPO_ROOT    = NOTEBOOK_DIR.parents[1]          # portfoliomodeling-gbm/
DATA_DIR     = REPO_ROOT / 'data'
RESULTS_DIR  = REPO_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Risk parameters ────────────────────────────────────────────────────────────
CONFIDENCE_LEVELS = [0.95, 0.99]   # 95 % and 99 %
TICKER = 'AAPL'                    # equity used in this notebook

print(f"Repo root : {REPO_ROOT}")
print(f"Data dir  : {DATA_DIR}")
print(f"Results   : {RESULTS_DIR}")

## 1. Data Loading

This cell attempts to load Nihan's cleaned CSV from `/data/`. If the file is not yet present (e.g. early in Week 1 before the data pipeline is complete), it falls back to downloading 5 years of daily closing prices directly via `yfinance`. Either way, the output is a clean `pd.Series` of closing prices named `prices`.

In [ ]:
csv_candidates = list(DATA_DIR.glob(f'{TICKER}*.csv')) + list(DATA_DIR.glob(f'{TICKER.lower()}*.csv'))

if csv_candidates:
    csv_path = csv_candidates[0]
    print(f"Loading data from Nihan's pipeline: {csv_path.name}")
    raw = pd.read_csv(csv_path, index_col=0, parse_dates=True)
    # Accept either a 'Close' or 'Adj Close' column
    close_col = 'Adj Close' if 'Adj Close' in raw.columns else 'Close'
    prices = raw[close_col].dropna()
    prices.name = TICKER
else:
    print("CSV not found — downloading via yfinance (fallback for fresh-kernel runs).")
    try:
        import yfinance as yf
        df = yf.download(TICKER, period='5y', auto_adjust=True, progress=False)
        prices = df['Close'].dropna()
        prices.name = TICKER
        # Cache locally so subsequent runs are faster
        prices.to_csv(DATA_DIR / f'{TICKER}_daily_close.csv', header=True)
        print(f"Downloaded {len(prices)} rows and cached to /data/{TICKER}_daily_close.csv")
    except Exception as e:
        raise RuntimeError(f"Could not load data. Put a CSV in /data/ or check your internet connection.\n{e}")

print(f"\nEquity  : {TICKER}")
print(f"Period  : {prices.index[0].date()} → {prices.index[-1].date()}")
print(f"Obs.    : {len(prices):,} trading days")
prices.tail()

## 2. Computing Log-Returns

### Formula

We define the **daily log-return** at time $t$ as:

$$
r_t = \ln\!\left(\frac{S_t}{S_{t-1}}\right)
$$

where $S_t$ is the closing price on day $t$.

**Why log-returns?**  
Log-returns are time-additive (multi-period returns are sums of single-period log-returns), approximately normally distributed for short horizons, and bounded below by $-\infty$ rather than $-1$, making them easier to work with in quantitative models. They are the standard input to GBM-based simulation.

**Implementation note:** `numpy.log(S_t / S_{t-1})` is equivalent to `numpy.diff(numpy.log(S))`, which is what `pandas.Series.pct_change()` followed by `numpy.log1p()` also computes. We use `numpy.log` directly for clarity.

In [ ]:
log_returns = np.log(prices / prices.shift(1)).dropna()
log_returns.name = f'{TICKER}_log_return'

print(f"Log-return series: {len(log_returns):,} observations")
print(f"\nDescriptive statistics:")
desc = log_returns.describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
print(desc.to_string())

# Annualised vol as a sanity check
ann_vol = log_returns.std() * np.sqrt(252)
print(f"\nAnnualised volatility (σ): {ann_vol:.2%}")

## 3. Value at Risk — Historical Simulation

### Definition

**Value at Risk at confidence level $\alpha$** answers the question:  
*"What is the minimum loss we would expect to exceed on the worst $(1-\alpha)$% of trading days?"*

Formally, for a return random variable $R$:

$$
\text{VaR}_\alpha = -\inf\{x \in \mathbb{R} : P(R \leq x) \geq 1-\alpha\}
$$

In plain terms, $\text{VaR}_{0.95}$ is the **5th percentile** of the return distribution (negated to express it as a positive loss). If $\text{VaR}_{0.95} = 0.025$, there is a 5% chance of losing more than 2.5% in a single day.

### Historical Simulation Method

Rather than fitting a parametric distribution, we:
1. **Sort** the empirical log-return series in ascending order.
2. **Read the $(1-\alpha)$-th quantile** — e.g. the 5th percentile for 95% VaR.
3. **Negate** the result so VaR is expressed as a positive loss magnitude.

$$
\widehat{\text{VaR}}_\alpha = -Q_{1-\alpha}(r_1, r_2, \ldots, r_T)
$$

where $Q_{1-\alpha}$ denotes the $(1-\alpha)$-th empirical quantile.

This is **non-parametric**: no distribution assumption is imposed. The tail behaviour is driven entirely by what was observed historically.

**Limitation:** Historical simulation assumes the past distribution is representative of the future. It may underestimate risk after structural breaks (e.g. financial crises, regime changes) if the historical window does not contain similar episodes.

In [ ]:
def historical_var(returns: pd.Series, confidence: float) -> float:
    """
    Compute historical simulation Value at Risk.

    Parameters
    ----------
    returns    : pd.Series of log-returns (daily)
    confidence : confidence level, e.g. 0.95 for 95% VaR

    Returns
    -------
    float : VaR expressed as a positive loss (e.g. 0.025 = 2.5% loss)
    """
    # The (1 - confidence) quantile of the empirical distribution
    quantile = np.quantile(returns, 1 - confidence)
    return -quantile   # negate → positive loss magnitude


# ── Compute VaR at both confidence levels ──────────────────────────────────────
var_results = {}
for cl in CONFIDENCE_LEVELS:
    var_results[cl] = historical_var(log_returns, cl)
    print(f"VaR ({cl:.0%}) = {var_results[cl]:.4f}  ({var_results[cl]:.2%} daily loss threshold)")

## 4. Expected Shortfall — Historical Simulation

### Definition

**Expected Shortfall (ES)**, also known as **Conditional VaR (CVaR)** or **Expected Tail Loss (ETL)**, answers:  
*"Given that we are in the worst $(1-\alpha)$% of days, what is the **average** loss?"*

ES is a **coherent risk measure** (satisfying sub-additivity, monotonicity, translation invariance, and positive homogeneity — see Artzner et al., 1999), unlike VaR which violates sub-additivity in general.

The integral form:

$$
\text{ES}_\alpha = -\frac{1}{1-\alpha}\int_0^{1-\alpha} Q_u(R) \, du
$$

For a discrete empirical distribution this simplifies to the **conditional mean of returns below the VaR threshold**:

$$
\widehat{\text{ES}}_\alpha = -\mathbb{E}\bigl[R \mid R < -\widehat{\text{VaR}}_\alpha\bigr]
= -\frac{1}{|\mathcal{T}|} \sum_{r_t \in \mathcal{T}} r_t
$$

where $\mathcal{T} = \{r_t : r_t < -\widehat{\text{VaR}}_\alpha\}$ is the set of returns worse than the VaR threshold.

**Interpretation:** ES always exceeds VaR in magnitude and captures the severity of losses in the tail, not just whether a threshold is breached. Basel III / FRTB requires banks to use ES rather than VaR for internal models.

**Implementation:** Filter returns below the VaR threshold (i.e. below the $(1-\alpha)$ quantile), then take the arithmetic mean and negate.

In [ ]:
def historical_es(returns: pd.Series, confidence: float) -> float:
    """
    Compute historical simulation Expected Shortfall (CVaR).

    Parameters
    ----------
    returns    : pd.Series of log-returns (daily)
    confidence : confidence level, e.g. 0.95 for 95% ES

    Returns
    -------
    float : ES expressed as a positive loss
    """
    var_threshold = -historical_var(returns, confidence)   # negative quantile value
    tail_returns  = returns[returns < var_threshold]       # returns worse than VaR
    if len(tail_returns) == 0:
        return np.nan
    return -tail_returns.mean()   # negate → positive loss magnitude


# ── Compute ES at both confidence levels ───────────────────────────────────────
es_results = {}
for cl in CONFIDENCE_LEVELS:
    es_results[cl] = historical_es(log_returns, cl)
    n_tail = (log_returns < -var_results[cl]).sum()
    print(f"ES  ({cl:.0%}) = {es_results[cl]:.4f}  ({es_results[cl]:.2%} avg loss in tail)  "
          f"[based on {n_tail} tail observations]")

## 5. Summary Table

The table below consolidates VaR and ES at both confidence levels alongside the count and percentage of observations that fall in each tail. This table will be incorporated into the paper's methodology section.

In [ ]:
n = len(log_returns)

rows = []
for cl in CONFIDENCE_LEVELS:
    var_val = var_results[cl]
    es_val  = es_results[cl]
    n_tail  = (log_returns < -var_val).sum()
    rows.append({
        'Confidence Level'      : f'{cl:.0%}',
        'VaR (daily, log-ret)'  : f'{var_val:.4f}',
        'VaR (%)'               : f'{var_val:.2%}',
        'ES (daily, log-ret)'   : f'{es_val:.4f}',
        'ES (%)'                : f'{es_val:.2%}',
        'Tail obs. (n)'         : n_tail,
        'Tail obs. (%)'         : f'{n_tail/n:.2%}',
    })

summary_df = pd.DataFrame(rows).set_index('Confidence Level')

print(f"{'='*70}")
print(f"  Historical Simulation Risk Metrics — {TICKER}")
print(f"  Sample: {prices.index[0].date()} → {prices.index[-1].date()}  |  n = {n:,} obs")
print(f"{'='*70}")
print(summary_df.to_string())
print(f"{'='*70}")

summary_df

## 6. Return Distribution Visualisation

### What We Are Plotting

The histogram below shows the **empirical distribution of daily log-returns** for `AAPL`. Two vertical dashed lines mark the negative VaR thresholds:

- **Red dashed line** — 95% VaR threshold (5th percentile)
- **Dark red / maroon dashed line** — 99% VaR threshold (1st percentile)

Returns to the **left** of each line constitute the tail used for the corresponding ES calculation. A normal distribution curve (fitted to the sample mean and standard deviation) is overlaid in grey to highlight fat-tail behaviour: if empirical returns were truly normal, the tail observations would be much less frequent than we observe.

### Why This Matters for the Paper

The histogram provides visual evidence for the **excess kurtosis** and **negative skewness** of equity returns — both key motivations for using Monte Carlo simulation with a calibrated GBM (and eventually fat-tailed alternatives like Student-t or jump-diffusion models) rather than a simple parametric normal model.

In [ ]:
from scipy import stats

# ── Figure setup ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('#f9f9f9')
ax.set_facecolor('#f9f9f9')

# ── Empirical histogram ────────────────────────────────────────────────────────
n_bins = 80
counts, bins, patches = ax.hist(
    log_returns,
    bins=n_bins,
    density=True,
    color='#1e3a5f',
    alpha=0.70,
    edgecolor='white',
    linewidth=0.3,
    label=f'{TICKER} empirical log-returns'
)

# Shade tail regions beyond each VaR threshold
tail_colors = {0.95: ('#e74c3c', 0.25), 0.99: ('#7b0000', 0.35)}
for cl, (col, alpha_val) in tail_colors.items():
    var_val = var_results[cl]
    for patch, left in zip(patches, bins[:-1]):
        if left < -var_val:
            patch.set_facecolor(col)
            patch.set_alpha(0.75)

# ── Normal distribution overlay ────────────────────────────────────────────────
x_range = np.linspace(log_returns.min() * 1.3, log_returns.max() * 1.3, 500)
mu_hat, sigma_hat = log_returns.mean(), log_returns.std()
normal_pdf = stats.norm.pdf(x_range, loc=mu_hat, scale=sigma_hat)
ax.plot(x_range, normal_pdf, color='#555555', linewidth=1.5, linestyle='--',
        alpha=0.8, label=f'Normal fit  (μ={mu_hat:.4f}, σ={sigma_hat:.4f})')

# ── VaR vertical lines ─────────────────────────────────────────────────────────
var_line_styles = {
    0.95: dict(color='#e74c3c', linewidth=2.0, linestyle='--', label=f'VaR 95% = {var_results[0.95]:.2%}'),
    0.99: dict(color='#7b0000', linewidth=2.0, linestyle=':',  label=f'VaR 99% = {var_results[0.99]:.2%}'),
}
for cl, style in var_line_styles.items():
    ax.axvline(x=-var_results[cl], **style)
    # Annotation arrow
    ax.annotate(
        f'VaR {cl:.0%}\n{var_results[cl]:.2%}',
        xy=(-var_results[cl], ax.get_ylim()[1] * 0.5 if ax.get_ylim()[1] > 0 else 10),
        xytext=(-var_results[cl] - 0.010, counts.max() * (0.65 if cl == 0.95 else 0.45)),
        fontsize=9,
        color=style['color'],
        ha='right',
        arrowprops=dict(arrowstyle='->', color=style['color'], lw=1.2)
    )

# ── ES annotation ──────────────────────────────────────────────────────────────
for cl in CONFIDENCE_LEVELS:
    es_val = es_results[cl]
    ax.axvline(x=-es_val, color=tail_colors[cl][0], linewidth=1.0,
               linestyle=':', alpha=0.6)

# ── Labels and styling ─────────────────────────────────────────────────────────
ax.set_title(
    f'Empirical Log-Return Distribution — {TICKER}\n'
    f'Historical Simulation VaR & Expected Shortfall | '
    f'{prices.index[0].strftime("%Y-%m-%d")} to {prices.index[-1].strftime("%Y-%m-%d")}',
    fontsize=13, fontweight='bold', pad=12
)
ax.set_xlabel('Daily Log-Return  $r_t = \\ln(S_t / S_{t-1})$', fontsize=11)
ax.set_ylabel('Probability Density', fontsize=11)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=1))
ax.legend(fontsize=9, framealpha=0.85, loc='upper left')
ax.grid(axis='y', alpha=0.3, linewidth=0.5)
ax.grid(axis='x', alpha=0.2, linewidth=0.5)

# ── Stats annotation box ───────────────────────────────────────────────────────
stats_text = (
    f'n = {len(log_returns):,}\n'
    f'μ = {mu_hat:.4f}\n'
    f'σ = {sigma_hat:.4f}\n'
    f'Skew = {log_returns.skew():.3f}\n'
    f'Kurt = {log_returns.kurt():.3f}'
)
ax.text(0.98, 0.97, stats_text, transform=ax.transAxes,
        fontsize=8.5, verticalalignment='top', horizontalalignment='right',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.75))

plt.tight_layout()

# ── Save BEFORE plt.show() — per project convention (Section 6a) ───────────────
fig_filename = f'{TICKER}_log_return_distribution_VaR_ES.png'
fig_path = RESULTS_DIR / fig_filename
fig.savefig(fig_path, dpi=300, bbox_inches='tight')
print(f"Figure saved → {fig_path}")

plt.show()
print("Figure saved successfully to /results/")

## 7. Interpretation and Discussion

This section interprets the numerical results and contextualises them for inclusion in the paper's **Methodology / Results** section.

In [ ]:
print("=" * 65)
print(f"  INTERPRETATION SUMMARY — {TICKER}")
print("=" * 65)

for cl in CONFIDENCE_LEVELS:
    var_val = var_results[cl]
    es_val  = es_results[cl]
    pct     = 1 - cl
    n_tail  = (log_returns < -var_val).sum()
    print(f"\n── {cl:.0%} Confidence Level ──")
    print(f"  VaR ({cl:.0%}): On the worst {pct:.0%} of trading days,")
    print(f"           we expect to lose at least {var_val:.2%} of portfolio value.")
    print(f"  ES  ({cl:.0%}): On those worst {pct:.0%} of days,")
    print(f"           the average loss is {es_val:.2%} of portfolio value.")
    print(f"  Tail obs: {n_tail} days out of {len(log_returns):,} exceeded the VaR threshold.")
    print(f"  ES / VaR ratio: {es_val / var_val:.2f}x  (>1 indicates fat-tail severity)")

print(f"\n── Excess Kurtosis ──")
kurt = log_returns.kurt()
skew = log_returns.skew()
print(f"  Kurtosis (excess): {kurt:.3f}  (Normal = 0; positive = fat tails)")
print(f"  Skewness         : {skew:.3f}  (Normal = 0; negative = left-skewed, more severe losses)")
if kurt > 0:
    print(f"  → Returns exhibit positive excess kurtosis: the tail is FATTER than Normal.")
    print(f"    This motivates using empirical historical simulation rather than a")
    print(f"    parametric Normal model, which would UNDERESTIMATE tail risk.")
print("=" * 65)

## 8. Reproducibility Checklist

Before committing this notebook to the `feature/risk-metrics` branch:

- [ ] **Kernel → Restart & Run All** — confirm zero errors from a fresh kernel
- [ ] **Clear outputs** before committing (`Kernel → Restart & Clear Output`)
- [ ] `/results/AAPL_log_return_distribution_VaR_ES.png` is present and saved at 300 dpi
- [ ] Commit message follows convention: `[risk] add historical simulation VaR ES notebook`
- [ ] Open a pull request into `dev` and request one review from Ayush (Lead Researcher)

---

## References

- Artzner, P., Delbaen, F., Eber, J.-M., & Heath, D. (1999). Coherent measures of risk. *Mathematical Finance*, 9(3), 203–228.
- Basel Committee on Banking Supervision (2019). *Minimum Capital Requirements for Market Risk* (FRTB). Bank for International Settlements.
- Hull, J. C. (2018). *Risk Management and Financial Institutions* (5th ed.). Wiley.
- McNeil, A. J., Frey, R., & Embrechts, P. (2015). *Quantitative Risk Management* (Rev. ed.). Princeton University Press.